In [ ]:
from langchain_ollama import ChatOllama
import numpy as np
import pandas as pd 
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
import sqlite3
from datetime import date
import matplotlib.pyplot as plt 
import seaborn as sns 

In [ ]:
conn = sqlite3.connect("MoneyWise.db")
cursor = conn.cursor()

In [ ]:
cursor.execute("SELECT * FROM Transactions")
rows = cursor.fetchall()

transactions_df = pd.DataFrame(rows, columns=[
    "id",
    "date",
    "title",
    "amount",
    "type",
    "category",
    "payment_method"
])

display(transactions_df)

In [ ]:
cursor.execute("SELECT * FROM Goals")
rows = cursor.fetchall()

goals_df = pd.DataFrame(rows, columns=[
    "id",
    "Title",
    "Started_at",
    "Deadline",
    "Target_Amount",
    "Saved_Amount",
    "Status"
])

display(goals_df)

In [ ]:
goals_df['Started_at'] = pd.to_datetime(goals_df['Started_at'])
goals_df['Deadline'] = pd.to_datetime(goals_df['Deadline'])

In [ ]:
transactions_df['date'] = pd.to_datetime(transactions_df['date'])


### Income vs expense

In [ ]:
       
while True:
    try:
        year = int(input("Select Year (e.g., 2024): "))
        if 1900 <= year <= 2100:  # Adjust the range as needed
            break
        print("Please enter a year between 1900 and 2100.")
    except ValueError:
        print("Invalid input. Please enter a numerical year.")

in_vs_ex_df = transactions_df[
    (transactions_df['date'].dt.year == year)
]

In [ ]:
in_vs_ex_df['month'] = in_vs_ex_df['date'].dt.strftime('%b')
plot_df = (
    in_vs_ex_df
    .groupby(['month', 'type'])['amount']
    .sum()
    .reset_index()
)

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

income = []
expense = []

for month in month_order:
    income_value = plot_df[
        (plot_df['month'] == month) &
        (plot_df['type'] == 'Income')
    ]['amount']

    expense_value = plot_df[
        (plot_df['month'] == month) &
        (plot_df['type'] == 'Expense')
    ]['amount']

    income.append(income_value.values[0] if not income_value.empty else 0)
    expense.append(expense_value.values[0] if not expense_value.empty else 0)

x = np.arange(len(month_order))
width = 0.35

fig, ax = plt.subplots(figsize=(12,5))

bars1 = ax.bar(
    x - width/2,
    income,
    width,
    label='Income'
)

bars2 = ax.bar(
    x + width/2,
    expense,
    width,
    label='Expense'
)

ax.bar_label(bars1, fmt='%.0f')
ax.bar_label(bars2, fmt='%.0f')

ax.set_xticks(x)
ax.set_xticklabels(month_order)

ax.set_title("Month-wise Income vs Expense")
ax.set_xlabel("Month")
ax.set_ylabel("Total Amount")

ax.legend()

plt.tight_layout()
plt.show()

### Category wise monthly expense donut chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

transactions_df['date'] = pd.to_datetime(transactions_df['date'])

while True:
    try:
        year = int(input("Select Year (e.g. 2024): "))
        
        if 1900 <= year <= 2100:
            break
            
        print("Enter a valid year.")
        
    except ValueError:
        print("Invalid input.")

while True:
    try:
        month = int(input("Select Month (1-12): "))
        
        if 1 <= month <= 12:
            break
            
        print("Please enter a value between 1 and 12.")
        
    except ValueError:
        print("Invalid input.")

month_df = transactions_df[
    (transactions_df['date'].dt.year == year) &
    (transactions_df['date'].dt.month == month) &
    (transactions_df['type'] == 'Expense')
]

category_expense = (
    month_df.groupby('category')['amount']
    .sum()
    .sort_values(ascending=False)
)

total_expense = category_expense.sum()

fig, ax = plt.subplots(figsize=(8,8))

def format_label(pct):
    amount = int(pct / 100 * total_expense)
    return f"{pct:.1f}%\n₹{amount}"

wedges, texts, autotexts = ax.pie(
    category_expense,
    labels=category_expense.index,
    autopct=format_label,
    startangle=90,
    wedgeprops=dict(width=0.4)
)

month_name = pd.to_datetime(str(month), format='%m').strftime('%B')

ax.set_title(
    f"{month_name} {year} Category-wise Expense Distribution"
)

plt.tight_layout()
plt.show()

### Savings Trend — Line Chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

transactions_df['date'] = pd.to_datetime(transactions_df['date'])

while True:
    try:
        year = int(input("Select Year: "))
        
        if 1900 <= year <= 2100:
            break
            
        print("Enter valid year.")
        
    except ValueError:
        print("Invalid input.")

year_df = transactions_df[
    transactions_df['date'].dt.year == year
].copy()

year_df['month'] = year_df['date'].dt.strftime('%b')

monthly_summary = (
    year_df.groupby(['month', 'type'])['amount']
    .sum()
    .unstack(fill_value=0)
)

monthly_summary['Savings'] = (
    monthly_summary['Income'] - monthly_summary['Expense']
)

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

monthly_summary = monthly_summary.reindex(month_order)

savings_df = monthly_summary.reset_index()

fig, ax = plt.subplots(figsize=(12,5))

sns.lineplot(
    data=savings_df,
    x='month',
    y='Savings',
    marker='o',
    linewidth=3,
    ax=ax
)

for i, value in enumerate(savings_df['Savings']):
    ax.text(i, value, f'₹{int(value)}')

ax.set_title(f"Savings Trend - {year}")
ax.set_xlabel("Month")
ax.set_ylabel("Savings")

plt.tight_layout()
plt.show()



### Goal Progress

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

goals_df['Progress'] = (
    goals_df['Saved_Amount'] / goals_df['Target_Amount']
) * 100

goals_df = goals_df.sort_values(by='Progress', ascending=True)

fig, ax = plt.subplots(figsize=(12,6))

sns.barplot(
    data=goals_df,
    x='Progress',
    y='Title',
    hue='Status',
    dodge=False,
    ax=ax
)

for index, value in enumerate(goals_df['Progress']):
    ax.text(
        value + 1,
        index,
        f'{value:.1f}%',
        va='center'
    )

ax.set_xlim(0, 100)

ax.set_title("Goal Progress Tracker")
ax.set_xlabel("Progress (%)")
ax.set_ylabel("Goals")

plt.tight_layout()
plt.show()